# 3) M2.3 European market design — Exercises

**Goals:** net positions and congestion income, your own multi-zone coupling LP checked against PyPSA, the value of each border's capacity, and who gains and who loses when markets are coupled.

The setup cell holds the lecture's three synthetic zones (SI, HR, HU), the isolated and base transfer capacities, and a PyPSA helper for checking. Flows are positive from the first zone of a border to the second. Prices in €/MWh, quantities in MW, money in €/h.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import logging
import warnings

import numpy as np
import pandas as pd
import pypsa
from scipy.optimize import linprog

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

ZONES = ["SI", "HR", "HU"]
OFFERS = pd.DataFrame([
    ("SI", "nuclear", 10, 700), ("SI", "hydro", 5, 500), ("SI", "coal", 95, 500), ("SI", "gas", 115, 300),
    ("HR", "hydro", 5, 900), ("HR", "wind", 0, 500), ("HR", "gas", 120, 700), ("HR", "ocgt", 160, 400),
    ("HU", "nuclear", 12, 1900), ("HU", "solar", 0, 800), ("HU", "gas", 100, 1500), ("HU", "ocgt", 150, 500),
], columns=["zone", "order", "price", "qty"])
BIDS = pd.DataFrame([
    ("SI", "load", 3000, 1300), ("HR", "load", 3000, 2400), ("HR", "industry", 130, 150),
    ("HU", "load", 3000, 3500), ("HU", "storage", 60, 300),
], columns=["zone", "order", "price", "qty"])
ISOLATED = {("SI", "HR"): 0, ("HR", "HU"): 0, ("SI", "HU"): 0}
BASE_ATC = {("SI", "HR"): 500, ("HR", "HU"): 400, ("SI", "HU"): 300}


def couple_pypsa(atc):
    """Zonal prices and welfare from PyPSA (zones as buses, borders as bidirectional links), for checking."""
    n = pypsa.Network()
    n.set_snapshots([0])
    n.add("Bus", ZONES)
    for r in OFFERS.itertuples():
        n.add("Generator", f"{r.zone}_{r.order}", bus=r.zone, p_nom=r.qty, marginal_cost=r.price)
    for r in BIDS.itertuples():
        n.add("Generator", f"{r.zone}_{r.order}", bus=r.zone, p_nom=r.qty, sign=-1, marginal_cost=-r.price)
    for (z0, z1), cap in atc.items():
        n.add("Link", f"{z0}-{z1}", bus0=z0, bus1=z1, p_nom=max(cap, 1e-6), p_min_pu=-1)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n.buses_t.marginal_price.iloc[0], -n.objective

### Warm-ups

**1. Net positions**

Write `net_positions(flows)` for a dict `{(zone0, zone1): flow}`: each zone's net export (MW, positive = exporting). Net positions of a closed system sum to zero.

In [2]:
def net_positions(flows: dict) -> pd.Series:
    """Net export (MW) of each zone from border flows."""
    ### BEGIN SOLUTION
    pos = pd.Series(0.0, index=ZONES)
    for (z0, z1), f in flows.items():
        pos[z0] += f
        pos[z1] -= f
    return pos
    ### END SOLUTION

In [3]:
lecture_flows = {("SI", "HR"): 500.0, ("HR", "HU"): -400.0, ("SI", "HU"): -100.0}
pos = net_positions(lecture_flows)
assert np.isclose(pos.sum(), 0) and np.isclose(pos["HR"], -900) and np.isclose(pos["SI"], 400) and np.isclose(pos["HU"], 500)
pos

SI    400.0
HR   -900.0
HU    500.0
dtype: float64

**2. Congestion income**

Write `congestion_income(prices, flows)` = Σ flow × (price at the receiving end − price at the sending end). Check it on the lecture's base result (18,000 €/h) and on equal prices (0).

In [4]:
def congestion_income(prices, flows: dict) -> float:
    """Congestion income (EUR/h) of all borders."""
    ### BEGIN SOLUTION
    return float(sum(f * (prices[z1] - prices[z0]) for (z0, z1), f in flows.items()))
    ### END SOLUTION

In [5]:
base_prices = pd.Series({"SI": 100.0, "HR": 120.0, "HU": 100.0})
assert np.isclose(congestion_income(base_prices, lecture_flows), 18_000)
assert np.isclose(congestion_income(pd.Series(115.0, index=ZONES), lecture_flows), 0), "no price difference, no income"

### Core

**3. Your own coupling LP**

Write `couple(atc)` returning a dict with zonal `prices` (Series), `flows` (dict), `welfare` (€/h), and accepted quantities `sold` and `bought` (arrays in the order of `OFFERS` and `BIDS`). Check it against PyPSA for the isolated case, the base case and five random sets of capacities.

In [6]:
def couple(atc: dict) -> dict:
    """Multi-zone welfare maximisation with transfer-capacity limits."""
    ### BEGIN SOLUTION
    borders = list(atc)
    n_o, n_b, n_l = len(OFFERS), len(BIDS), len(borders)
    A = np.zeros((len(ZONES), n_o + n_b + n_l))
    for k, z in enumerate(ZONES):
        A[k, :n_o] = (OFFERS.zone == z).astype(float)
        A[k, n_o:n_o + n_b] = -(BIDS.zone == z).astype(float)
        for j, (z0, z1) in enumerate(borders):
            A[k, n_o + n_b + j] = -1.0 if z == z0 else (1.0 if z == z1 else 0.0)
    bounds = (list(zip(np.zeros(n_o), OFFERS.qty)) + list(zip(np.zeros(n_b), BIDS.qty))
              + [(-atc[b], atc[b]) for b in borders])
    res = linprog(np.r_[OFFERS.price, -BIDS.price, np.zeros(n_l)], A_eq=A, b_eq=np.zeros(len(ZONES)),
                  bounds=bounds, method="highs")
    return {"prices": pd.Series(res.eqlin.marginals, index=ZONES),
            "flows": dict(zip(borders, res.x[n_o + n_b:])), "welfare": -res.fun,
            "sold": res.x[:n_o], "bought": res.x[n_o:n_o + n_b]}
    ### END SOLUTION

In [7]:
rng = np.random.default_rng(3)
cases = [ISOLATED, BASE_ATC] + [{b: float(rng.integers(0, 1200)) for b in BASE_ATC} for _ in range(5)]
for atc in cases:
    mine = couple(atc)
    p_ref, w_ref = couple_pypsa(atc)
    assert np.allclose(mine["prices"].to_numpy(), p_ref.to_numpy()) and np.isclose(mine["welfare"], w_ref), atc
    assert np.isclose(net_positions(mine["flows"]).sum(), 0)
print(f"own coupling LP = PyPSA in all {len(cases)} cases")

own coupling LP = PyPSA in all 7 cases


**4. What is each border worth?**

Write `border_values(atc, h=0.5)` returning, for every border, the welfare gained per extra MW of its capacity (central finite difference), and compare with the price difference across that border in the base case.

In [8]:
def border_values(atc: dict, h: float = 0.5) -> pd.DataFrame:
    """Marginal welfare value (EUR/MWh) of each border's capacity, and the absolute price difference across it."""
    ### BEGIN SOLUTION
    prices = couple(atc)["prices"]
    rows = {}
    for b in atc:
        up = couple({**atc, b: atc[b] + h})["welfare"]
        down = couple({**atc, b: atc[b] - h})["welfare"]
        rows[f"{b[0]}-{b[1]}"] = {"value_per_MW": (up - down) / (2 * h),
                                  "price_difference": abs(prices[b[1]] - prices[b[0]])}
    return pd.DataFrame(rows).T
    ### END SOLUTION

In [9]:
values = border_values(BASE_ATC)
assert np.allclose(values.value_per_MW, values.price_difference, atol=1e-6), "envelope theorem on every border"
assert np.isclose(values.loc["SI-HR", "value_per_MW"], 20) and np.isclose(values.loc["SI-HU", "value_per_MW"], 0)
values.round(3)

       value_per_MW  price_difference
SI-HR          20.0              20.0
HR-HU          20.0              20.0
SI-HU           0.0               0.0

**5. Interpretation**

The Slovenia–Hungary border carries power in the base case, yet one more MW on it is worth nothing. Why? And why does the value of the Croatian borders equal the price difference across them — what would a TSO compare this number with before building a new interconnector?

<!-- BEGIN SOLUTION -->
Slovenia and Hungary end up at the same price (100 €/MWh), so the SI–HU border is not binding: it carries 100 MW from Hungary to Slovenia, well below its 300 MW limit, and an extra MW of capacity would not change any dispatch. By complementary slackness its multiplier — the value of more capacity — is zero, which is exactly the zero price difference. The Croatian borders are full; one more MW on either lets one more MW of 100 €/MWh power from a neighbour replace 120 €/MWh Croatian gas, saving 20 €/h — the price difference, by the envelope theorem (Maths-3). A TSO would compare the value of extra capacity, summed over all hours of the year and over the asset's lifetime (most hours will not be congested, and the value falls as capacity grows, as the lecture's large-ATC case showed), with the annualised cost of the interconnector per MW. This is the core of cost-benefit analyses for cross-border projects, and Module 05 does it with full-year models.
<!-- END SOLUTION -->

### Challenge

**6. Who gains and who loses from coupling?**

Write `surplus_by_zone(result)` returning, for each zone, consumer surplus and producer surplus (€/h) at that zone's price. Compare isolation with the base coupling: the change in each zone's consumer and producer surplus, and check that the total change plus congestion income equals the welfare gain.

In [10]:
def surplus_by_zone(result: dict) -> pd.DataFrame:
    """Consumer and producer surplus (EUR/h) per zone at the zonal prices."""
    ### BEGIN SOLUTION
    p_off = OFFERS.zone.map(result["prices"]).to_numpy()
    p_bid = BIDS.zone.map(result["prices"]).to_numpy()
    producer = pd.Series((p_off - OFFERS.price.to_numpy()) * result["sold"]).groupby(OFFERS.zone).sum()
    consumer = pd.Series((BIDS.price.to_numpy() - p_bid) * result["bought"]).groupby(BIDS.zone).sum()
    return pd.DataFrame({"consumer": consumer, "producer": producer}).reindex(ZONES).fillna(0.0)
    ### END SOLUTION

In [11]:
iso, cpl = couple(ISOLATED), couple(BASE_ATC)
change = surplus_by_zone(cpl) - surplus_by_zone(iso)
ci = congestion_income(cpl["prices"], cpl["flows"])
assert np.isclose(change.to_numpy().sum() + ci, cpl["welfare"] - iso["welfare"]), "gains + congestion income = welfare gain"
assert change.loc["HR", "consumer"] > 0, "Croatian consumers gain from cheaper imports"
assert change.loc["SI", "consumer"] < 0, "Slovenian consumers pay more after coupling"
assert change.loc["SI", "producer"] > 0 and change.loc["HR", "producer"] < 0
print(f"welfare gain {cpl['welfare'] - iso['welfare']:,.0f} EUR/h, of which congestion income {ci:,.0f} EUR/h")
change.round(0)

welfare gain 33,500 EUR/h, of which congestion income 18,000 EUR/h


      consumer  producer
zone                    
SI     -6500.0    8500.0
HR     97500.0  -84000.0
HU         0.0       0.0

**7. Interpretation**

Coupling raises total welfare, but not everyone gains. Describe who wins and who loses in each zone, explain why the losers' losses are smaller than the winners' gains, and say why this distribution matters for the politics of market integration and new interconnectors.

<!-- BEGIN SOLUTION -->
In Croatia (the importer) the price falls from 160 to 120 €/MWh: Croatian consumers gain a lot, Croatian producers lose — the OCGT no longer runs and every other Croatian plant earns 40 €/MWh less. In Slovenia (an exporter) the price rises from 95 to 100 €/MWh: Slovenian producers gain from selling more at a higher price, but Slovenian consumers now pay more for the same power. In Hungary the price stays at 100 €/MWh and the extra exports come from gas plants that are paid exactly their cost, so neither Hungarian consumers nor producers change — exporting does not always pay. The losses are smaller than the gains because coupling replaces expensive Croatian generation (OCGT at 160 €/MWh, then part of the gas at 120) with cheaper foreign power; that real cost saving is the net welfare gain (33,500 €/h). It is split between the surplus changes of consumers and producers (15,500 €/h in total, after the transfers between them cancel out) and the congestion income collected on the full borders (18,000 €/h). Politically, the losers are concentrated and visible (consumers in exporting countries facing higher bills, producers in importing countries losing market share) while the gains are spread, so integration often meets resistance even when it is clearly beneficial overall. That is why cost-benefit analyses of interconnectors report the distribution between countries and groups, and why compensation mechanisms and the use of congestion income are part of the debate.
<!-- END SOLUTION -->